# gramlab — overnight run: grammar-heads comparison + bigger model on clean Wikipedia text

Runs unattended via **Save Version → Save & Run All**. Settings: GPU T4 x2, Internet on. Budget ≈ 4–5 h; a time guard stops the big training at 4 h so the job stays far below Kaggle's 12 h limit.

Sources: AMALGUM (Wikipedia bios, Wikinews, Wikivoyage, wikiHow, academic, fiction, interviews), Simple English Wikipedia (all), English Wikipedia (first ~120k articles). Evaluation on gold GUM text, never trained on. No forum, email or review text.

## 0. Setup

In [ ]:
import os, subprocess, sys, time
T0 = time.time()
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    subprocess.run(['git','clone','--depth','1','https://github.com/furevych/gramlab.git'], check=True)
else:
    subprocess.run(['git','-C','gramlab','pull','--ff-only'], check=False)
os.chdir('/kaggle/working/gramlab')
os.makedirs('external', exist_ok=True)
for url, d in [('https://github.com/gucorpling/amalgum','external/amalgum'),
               ('https://github.com/UniversalDependencies/UD_English-GUM','external/UD_English-GUM'),
               ('https://github.com/alexwarstadt/blimp','external/blimp')]:
    if not os.path.exists(d):
        subprocess.run(['git','clone','--depth','1',url,d], check=True, env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
import torch, tokenizers, datasets
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '',
      '| bf16', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None, '| tokenizers', tokenizers.__version__, '| datasets', datasets.__version__)
print(os.getcwd(), sorted(os.listdir('.')))

## 1. Tokenizer + shards for the small corpora

In [ ]:
!python -m gramlab.prepare --config configs/tiny.json

## 2. Experiment A — baseline tiny model (≈13 min)

In [ ]:
!python -m gramlab.train --config configs/tiny.json

## 3. Experiment B — same model + grammar heads (UPOS + chunk tags predicted at train time only) (≈13 min)

In [ ]:
!python -m gramlab.train --config configs/tiny_upos.json

## 4. Compare A vs B: bits per byte on gold text + BLiMP grammaticality (67 phenomena, 300 pairs each)

In [ ]:
import json, subprocess
res = {}
for run in ['tiny', 'tiny_upos']:
    out = subprocess.run([sys.executable, '-m', 'gramlab.evaluate', '--ckpt', f'work/runs/{run}/ckpt.pt', '--corpus', 'gum_test',
                          '--blimp', 'external/blimp/data', '--blimp_limit', '300'], capture_output=True, text=True)
    print(out.stderr[-2000:] if out.returncode else '', flush=True)
    res[run] = json.loads(out.stdout[out.stdout.index('{'):])
    print(run, {k: v for k, v in res[run].items() if k != 'blimp'}, 'BLiMP mean', round(res[run]['blimp']['_mean'], 4), flush=True)
os.makedirs('results', exist_ok=True); json.dump(res, open('results/compare_tiny_vs_upos.json', 'w'), indent=1)
a, b = res['tiny'], res['tiny_upos']
print(f"\nbits/byte  baseline {a['eval_bits_per_byte']:.4f}  grammar-heads {b['eval_bits_per_byte']:.4f}")
print(f"BLiMP mean baseline {a['blimp']['_mean']:.4f}  grammar-heads {b['blimp']['_mean']:.4f}")
diffs = sorted(((b['blimp'][k] - a['blimp'][k], k) for k in a['blimp'] if not k.startswith('_')), reverse=True)
print('largest BLiMP gains for grammar heads:', [(k, round(d, 3)) for d, k in diffs[:5]])
print('largest BLiMP losses:', [(k, round(d, 3)) for d, k in diffs[-5:]])

## 5. More training data: Simple English Wikipedia (all) + first 120k English Wikipedia articles (≈15–25 min download + tokenization)

In [ ]:
!python scripts/fetch_wikipedia.py --lang simple --out data/simplewiki.jsonl
!python scripts/fetch_wikipedia.py --lang en --max_articles 120000 --out data/enwiki.jsonl
!python -m gramlab.prepare --config configs/small_wiki.json --corpus simplewiki --corpus enwiki
import json
for c in ['amalgum', 'simplewiki', 'enwiki']:
    m = json.load(open(f'work/bpe16000/shards/{c}/meta.json')); print(c, f"{m['n_words']:,} words -> {m['n_tokens']:,} tokens")

## 6. Train the small model (≈25M params, ctx 256) on the Wikipedia mix — time-guarded at 4 h

In [ ]:
!python -m gramlab.train --config configs/small_wiki.json

## 7. Fine-tune it on the synthetic Q&A so it answers in turns, then evaluate

In [ ]:
!python scripts/make_wiki_qa.py --amalgum external/amalgum/amalgum --out data/wiki_qa.jsonl
!python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus wiki_qa --out small_wiki_sft --epochs 3
!python -m gramlab.evaluate --ckpt work/runs/small_wiki/ckpt.pt --corpus gum_test --blimp external/blimp/data --blimp_limit 300

## 8. What the model says now (fixed prompts, deterministic-ish sampling)

In [ ]:
import sys, os, torch
sys.path.insert(0, '/kaggle/working/gramlab'); os.chdir('/kaggle/working/gramlab')
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
CANDIDATES = ['work/runs/small_wiki_sft/ckpt.pt', 'work/runs/small_wiki/ckpt.pt', 'work/runs/tiny_sft/ckpt.pt', 'work/runs/tiny/ckpt.pt']
CKPT = next(p for p in CANDIDATES if os.path.exists(p))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tok, state = load_checkpoint(CKPT, dev)
MODE = 'chat' if state.get('sft') else 'base'
print(f'loaded {CKPT}: {model.n_params()/1e6:.1f}M non-embedding params, mode={MODE}')
history = []
def ask(text, temperature=0.5, top_k=40, top_p=0.9, rep=1.2, max_new=80, reset=False):
    global history
    if reset: history = []
    if MODE == 'chat':
        history.append(('user', text)); ids = []
        for role, t in history[-6:]:
            ids += [USER if role == 'user' else ASSISTANT] + tok.encode(t) + [END]
        ids += [ASSISTANT]; stop = (END, EOS)
    else:
        ids = tok.encode(text); stop = ()
    x = torch.tensor([ids[-(model.ctx - 1):]], dtype=torch.long, device=dev)
    out = model.generate(x, max_new_tokens=max_new, temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=rep, stop_ids=stop)
    reply = tok.decode(out[0, x.size(1):].tolist())
    if MODE == 'chat': history.append(('assistant', reply))
    return reply
PROMPTS = ['Tell me about Belgrade.', 'Tell me about Paris.', 'Who was Napoleon?', 'What is the capital of Japan?',
           'Tell me about the Danube.', 'How do I bake bread?', 'Who was Albert Einstein?', 'What is photosynthesis?']
for p in PROMPTS:
    print('Q:', p); print('A:', ask(p, reset=True)); print()

## 9. Chat box (for interactive sessions)

In [ ]:
import ipywidgets as w
from IPython.display import display
box = w.Text(placeholder='say something...', layout=w.Layout(width='70%'), continuous_update=False)
btn = w.Button(description='Send'); out = w.Output()
def on_send(_=None):
    text = box.value.strip()
    if not text: return
    box.value = ''
    with out:
        print('you>   ', text); print('model> ', ask(text)); print()
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)

## 10. Keep the output small: drop the git clones and raw dumps (shards + checkpoints + results stay)

In [ ]:
import shutil, os, time
for d in ['external', 'data/enwiki.jsonl', 'data/simplewiki.jsonl', '.git']:
    p = os.path.join('/kaggle/working/gramlab', d)
    if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True)
    elif os.path.exists(p): os.remove(p)
os.system('du -sh /kaggle/working/gramlab/work/* 2>/dev/null')
print(f'total run time {(time.time()-T0)/3600:.2f} h')